# Module 2 Project Solution: CNN Intuition and Vision Transformer Attention


> **Colab setup:** This solution notebook uses only the default open-source Python stack available in Google Colab: NumPy, Matplotlib, and scikit-learn. No `pip install`, no API keys, and no external authorization are required.
>
> Running all cells produces the complete reference output, including plots and metrics.


## Project map
This project connects CNN locality with Transformer global token interaction using small images and transparent NumPy code.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

np.random.seed(7)
digits = load_digits()
img = digits.images[0] / 16.0
plt.imshow(img, cmap="gray")
plt.title("Input digit")
plt.axis("off")
plt.show()


In [ ]:
def conv2d_valid(image, kernel):
    kh, kw = kernel.shape
    out_h = image.shape[0] - kh + 1
    out_w = image.shape[1] - kw + 1
    out = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            patch = image[i:i+kh, j:j+kw]
            out[i, j] = np.sum(patch * kernel)
    return out

sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])
blur = np.ones((3, 3)) / 9
edge_x = conv2d_valid(img, sobel_x)
blurred = conv2d_valid(img, blur)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(img, cmap="gray"); axes[0].set_title("input")
axes[1].imshow(edge_x, cmap="coolwarm"); axes[1].set_title("Sobel x")
axes[2].imshow(blurred, cmap="gray"); axes[2].set_title("blur")
for ax in axes: ax.axis("off")
plt.show()


In [ ]:
def conv_params(kernel_size, in_channels, out_channels):
    return kernel_size * kernel_size * in_channels * out_channels + out_channels

def depthwise_separable_params(kernel_size, in_channels, out_channels):
    depthwise = kernel_size * kernel_size * in_channels
    pointwise = in_channels * out_channels
    bias = out_channels
    return depthwise + pointwise + bias

standard = conv_params(3, 32, 64)
separable = depthwise_separable_params(3, 32, 64)

plt.bar(["standard conv", "depthwise separable"], [standard, separable])
plt.title("Parameter budget comparison")
plt.ylabel("parameters")
plt.show()
print("standard:", standard, "separable:", separable, "ratio:", separable / standard)


In [ ]:
def patchify(image, patch_size):
    patches = []
    for i in range(0, image.shape[0], patch_size):
        for j in range(0, image.shape[1], patch_size):
            patch = image[i:i+patch_size, j:j+patch_size]
            patches.append(patch.reshape(-1))
    return np.array(patches)

patches = patchify(img, 2)
print("patches:", patches.shape)
plt.figure(figsize=(8, 2))
for k, p in enumerate(patches[:8]):
    plt.subplot(1, 8, k+1)
    plt.imshow(p.reshape(2, 2), cmap="gray")
    plt.axis("off")
plt.show()


In [ ]:
Z = patches
d_model = 6
Wq = np.random.randn(Z.shape[1], d_model) * 0.2
Wk = np.random.randn(Z.shape[1], d_model) * 0.2
Wv = np.random.randn(Z.shape[1], d_model) * 0.2

def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / exp_x.sum(axis=axis, keepdims=True)

Q = Z @ Wq
K = Z @ Wk
V = Z @ Wv
scores = (Q @ K.T) / np.sqrt(d_model)
A = softmax(scores, axis=1)
Y = A @ V

plt.imshow(A, cmap="magma")
plt.colorbar()
plt.title("Patch-to-patch attention matrix")
plt.xlabel("key patch")
plt.ylabel("query patch")
plt.show()
print("attention output:", Y.shape)
